In [1]:
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau

from models.vae_gpt import *
from train.trainer.Trainer_VAE_GPT import Trainer
from data.preprocessing.pipeline import Pipeline
from data.datasets.universal_dataset import CVADataset
from data.preprocessing.splitter import select_test_inh
from utils.paths import get_project_path

import os

In [2]:
def load_data(drop_inhib: str):
    prep = Pipeline(
        num_cycle=[1, 2, 3, 4], 
        inhibitor_name="all", 
        split="all",
        norm_feat=True
    )
    data = prep.full_data

    train_data, valid_data = select_test_inh(data, drop_inhib)
    
    return train_data, valid_data

In [3]:
INHIBITOR_NAME = "2-mercaptobenzimidazole"
normalize = True

train, val = load_data(drop_inhib=INHIBITOR_NAME)

train_dataset = CVADataset(train, normalize=normalize)
test_dataset = CVADataset(val, normalize=normalize)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [4]:
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = "cuda:1"
elif torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"

vae = VAE_GPT(
    cond_dim=41,
    seq_len=968,
    d_model=64,
    nhead=2,
    num_layers=2,
    dropout=0.15
).to(device)

opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)
scheduler = StepLR(opt_vae, step_size=25, gamma=0.1)

train_denormalize_fn = None
val_denormalize_fn = None

if normalize:
    train_denormalize_fn = train_dataset.denormalize
    val_denormalize_fn = test_dataset.denormalize

/home/smirnov@dohod.local/Desktop/electrochem-digital-twins/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [5]:
num_epoch = 100

trainer = Trainer(
    model=vae,
    loss_fn=vae_loss,
    epochs=num_epoch,
    optimizer=opt_vae,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    path_to_save_plots=os.path.join(get_project_path(), "reports", "gpt", INHIBITOR_NAME),
    path_to_save_models=os.path.join(get_project_path(), "models", "gpt", INHIBITOR_NAME, "best_model.pt"),
    path_to_save_tables=os.path.join(get_project_path(), "reports", "gpt", INHIBITOR_NAME),
    scheduler=scheduler,
    train_denorm_fn=train_denormalize_fn,
    val_denorm_fn=val_denormalize_fn,
    seed=42
)

In [6]:
trainer.train_model()

Epoch 000 — Train Loss: 0.447885, Val Loss: 0.285778
Epoch 001 — Train Loss: 0.244623, Val Loss: 0.216426
Epoch 002 — Train Loss: 0.182629, Val Loss: 0.163357
Epoch 003 — Train Loss: 0.150984, Val Loss: 0.138965
Epoch 004 — Train Loss: 0.117513, Val Loss: 0.125825
Epoch 005 — Train Loss: 0.102625, Val Loss: 0.114882
Epoch 006 — Train Loss: 0.098597, Val Loss: 0.114103
Epoch 007 — Train Loss: 0.096644, Val Loss: 0.113669
Epoch 008 — Train Loss: 0.095527, Val Loss: 0.112775
Epoch 009 — Train Loss: 0.094859, Val Loss: 0.113966
Epoch 010 — Train Loss: 0.094170, Val Loss: 0.112269
Epoch 011 — Train Loss: 0.093956, Val Loss: 0.113687
Epoch 012 — Train Loss: 0.093305, Val Loss: 0.113822
Epoch 013 — Train Loss: 0.092727, Val Loss: 0.115725
Epoch 014 — Train Loss: 0.092246, Val Loss: 0.115106
Epoch 015 — Train Loss: 0.091738, Val Loss: 0.120784
Epoch 016 — Train Loss: 0.090306, Val Loss: 0.116521
Epoch 017 — Train Loss: 0.089030, Val Loss: 0.112609
Epoch 018 — Train Loss: 0.088558, Val Loss: 0.